<a href="https://colab.research.google.com/github/etcex2969-spec/-AIFFEL_quest_eng/blob/main/PIPLINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                              RAG Pipeline — Dataset Benchmark & Evaluation

dataset_experiments.py
│
├── SECTION 1 — 검색(Retrieval) 품질 측정
│   ├── 1-A  MS MARCO         load_msmarco()           run_msmarco_experiment()
│   ├── 1-B  BEIR Benchmark   load_beir_subset()       run_beir_experiment()
│   └── 1-C  HotpotQA         load_hotpotqa()          run_hotpotqa_experiment()
│
├── SECTION 2 — 도메인 특화
│   ├── 2-A  FinQA             load_finqa()             run_finqa_experiment()
│   ├── 2-B  CodeSearchNet     load_codesearchnet()     run_codesearchnet_experiment()
│   └── 2-C  PubMedQA          load_pubmedqa()          run_pubmedqa_experiment()
│
└── SECTION 3 — 공통 평가 유틸
    ├── exact_match()          EM 계산
    ├── f1_score()             Token F1 계산
    ├── recall_at_k()          Recall@K 계산
    ├── save_results_csv()     결과 CSV 저장
    └── print_summary()        요약 출력

In [ ]:
pip install datasets openai

In [ ]:
from dataset_experiments import load_finqa, run_finqa_experiment, print_summary

# 실험 실행
results = run_finqa_experiment(pipeline, sample_size=100)

# 요약 출력
print_summary(results, "FinQA")

# CSV 저장
save_results_csv(results, "finqa_result.csv")

In [ ]:
# CELL 3 교체 예시
from rag_pipeline import RAGConfig, RAGPipeline

config   = RAGConfig(...)
pipeline = RAGPipeline(config, rag_client, llm_client)

In [ ]:
# KorQuAD — 한국어 RAG 실험 가장 추천!
from datasets import load_dataset

ds = load_dataset("squad_kor_v1", split="validation[:200]")

for item in ds:
    question = item["question"]        # RAGPipeline.run()에 넣을 질문
    context  = item["context"]         # RAG_CONTEXT_SNIPPET 대신 사용
    answer   = item["answers"]["text"] # 정답 비교용

    result = pipeline.run(question)
    print(f"Q: {question}")
    print(f"정답: {answer[0]}")
    print(f"모델: {result}")
    print("---")

In [ ]:
!pip install -q structlog pydantic tenacity requests openai

In [ ]:
"""
Enterprise-grade RAG Pipeline
개선사항: OOP + DI, tenacity 재시도, 고도화 프롬프트, 구조적 로깅 + request_id 추적
"""
from __future__ import annotations

import uuid
import logging
from typing import Optional, Protocol, runtime_checkable

import requests
import structlog
from pydantic import BaseModel, Field, field_validator
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

# ---------------------------------------------------------------------------
# 1. 설정 (Pydantic) — 환경 변수에서 자동 로드
# ---------------------------------------------------------------------------

class RAGConfig(BaseModel):
    """모든 설정을 한 곳에서 관리. 환경 변수 또는 직접 주입 가능."""

    rag_api_base: str = Field(
        default="https://rag-tool.example.com/api",
        validation_alias="RAG_API_BASE",
    )
    rag_api_key: str = Field(default="", validation_alias="RAG_API_KEY")
    rag_tool_name: str = Field(
        default="retrieval_tool", validation_alias="RAG_TOOL_NAME"
    )
    rag_context_snippet: str = Field(
        default="[RAG 지식] 기본 컨텍스트입니다.",
        validation_alias="RAG_CONTEXT_SNIPPET",
    )
    openai_api_key: str = Field(default="", validation_alias="OPENAI_API_KEY")
    openai_model: str = Field(default="gpt-4o-mini", validation_alias="OPENAI_MODEL")

    # 재시도 설정
    retry_attempts: int = Field(default=3, validation_alias="RAG_RETRY_COUNT")
    retry_min_wait: float = Field(default=1.0, validation_alias="RAG_BACKOFF_SEC")
    retry_max_wait: float = Field(default=10.0, validation_alias="RAG_MAX_BACKOFF_SEC")
    timeout_sec: float = Field(default=5.0, validation_alias="RAG_TIMEOUT_SEC")

    model_config = {"populate_by_name": True}

    # ── [추가] 유효성 검증 ──────────────────────────────────────
    @field_validator("retry_attempts")
    @classmethod
    def _validate_retry(cls, v: int) -> int:
        if v < 0:
            raise ValueError(f"retry_attempts 는 0 이상이어야 합니다. (입력값: {v})")
        return v

    @field_validator("timeout_sec")
    @classmethod
    def _validate_timeout(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"timeout_sec 는 0보다 커야 합니다. (입력값: {v})")
        return v

    @field_validator("rag_api_base")
    @classmethod
    def _validate_api_base(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("rag_api_base 는 빈 문자열일 수 없습니다.")
        return v

    @classmethod
    def from_env(cls) -> "RAGConfig":
        """환경 변수에서 설정을 로드합니다."""
        import os
        return cls(
            RAG_API_BASE=os.getenv("RAG_API_BASE", "https://rag-tool.example.com/api"),
            RAG_API_KEY=os.getenv("RAG_API_KEY", ""),
            RAG_TOOL_NAME=os.getenv("RAG_TOOL_NAME", "retrieval_tool"),
            RAG_CONTEXT_SNIPPET=os.getenv(
                "RAG_CONTEXT_SNIPPET", "[RAG 지식] 기본 컨텍스트입니다."
            ),
            OPENAI_API_KEY=os.getenv("OPENAI_API_KEY", ""),
            OPENAI_MODEL=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            RAG_RETRY_COUNT=int(os.getenv("RAG_RETRY_COUNT", "3")),
            RAG_BACKOFF_SEC=float(os.getenv("RAG_BACKOFF_SEC", "1.0")),
            RAG_MAX_BACKOFF_SEC=float(os.getenv("RAG_MAX_BACKOFF_SEC", "10.0")),
            RAG_TIMEOUT_SEC=float(os.getenv("RAG_TIMEOUT_SEC", "5.0")),
        )


# ---------------------------------------------------------------------------
# 2. 구조적 로깅 (structlog) — JSON 포맷, request_id 컨텍스트 바인딩
# ---------------------------------------------------------------------------

def _configure_logging() -> None:
    """
    [수정] 모듈 임포트 시 전역 로그 설정이 즉시 적용되는 부작용을 방지.
    테스트 환경에서는 호출하지 않아도 되며, 운영 엔트리포인트에서만 호출한다.
    """
    structlog.configure(
        processors=[
            structlog.contextvars.merge_contextvars,
            structlog.processors.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.JSONRenderer(),
        ],
        wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
    )

logger = structlog.get_logger()


# ---------------------------------------------------------------------------
# 3. 클라이언트 인터페이스 (Protocol) — 의존성 주입(DI) 기반
# ---------------------------------------------------------------------------

@runtime_checkable
class RAGClientProtocol(Protocol):
    def retrieve(self, context: str, question: str) -> str: ...


@runtime_checkable
class LLMClientProtocol(Protocol):
    def infer(self, prompt: str, temperature: float = 0.1) -> str: ...


# ---------------------------------------------------------------------------
# 4. RAG 클라이언트 — tenacity 재시도 + 세분화된 예외 처리
# ---------------------------------------------------------------------------

class RAGClient:
    """외부 RAG REST API 호출 클라이언트."""

    def __init__(self, config: RAGConfig) -> None:
        self._config = config
        self._log = logger.bind(component="RAGClient")
        # [수정] retry 데코레이터를 __init__ 에서 한 번만 생성 → 매 호출 재생성 방지
        self._retrying_call = retry(
            stop=stop_after_attempt(config.retry_attempts + 1),
            wait=wait_exponential(
                multiplier=config.retry_min_wait,
                max=config.retry_max_wait,
            ),
            retry=retry_if_exception_type(
                (requests.exceptions.Timeout, requests.exceptions.ConnectionError)
            ),
            before_sleep=before_sleep_log(
                logging.getLogger("tenacity"), logging.WARNING
            ),
            reraise=True,
        )(self._call_once)

    def retrieve(self, context: str, question: str) -> str:
        """tenacity 재시도 래퍼를 통해 RAG API 를 호출한다."""
        return self._retrying_call(context, question)

    def _call_once(self, context: str, question: str) -> str:
        """[수정] 실제 HTTP 호출 로직을 별도 메서드로 분리. 재시도 대상은 이 메서드."""
        cfg = self._config
        url = f"{cfg.rag_api_base}/rag"
        headers = {
            "Authorization": f"Bearer {cfg.rag_api_key}" if cfg.rag_api_key else "",
            "Content-Type": "application/json",
        }
        payload = {
            "context": context,
            "question": question,
            "options": {"include_sources": True},
        }
        self._log.info("rag_api_call", url=url)
        resp = requests.post(
            url, json=payload, headers=headers, timeout=cfg.timeout_sec
        )
        # 401/403 → 즉시 중단 (재시도 불가)
        if resp.status_code in (401, 403):
            raise PermissionError(
                f"RAG API 인증 실패: {resp.status_code}. 즉시 중단."
            )
        resp.raise_for_status()
        data = resp.json()
        return data.get("context", context)



# ---------------------------------------------------------------------------
# 5. LLM 클라이언트 — OpenAI 호출 래퍼
# ---------------------------------------------------------------------------

class OpenAILLMClient:
    """OpenAI ChatCompletion 호출 클라이언트."""

    def __init__(self, config: RAGConfig) -> None:
        self._config = config
        self._log = logger.bind(component="OpenAILLMClient")

    def infer(self, prompt: str, temperature: float = 0.1) -> str:
        if not self._config.openai_api_key:
            self._log.warning("openai_key_missing", fallback="simulation_mode")
            return f"[시뮬레이션] 프롬프트 수신 완료 (temperature={temperature})"

        try:
            from openai import OpenAI  # type: ignore
            client = OpenAI(api_key=self._config.openai_api_key)
            self._log.info("llm_infer_start", model=self._config.openai_model)
            response = client.chat.completions.create(
                model=self._config.openai_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=1024,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            self._log.error("llm_infer_failed", error=str(exc))
            raise


# ---------------------------------------------------------------------------
# 6. 프롬프트 빌더 — Persona + Constraint + Few-shot
# ---------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = """
[예시 1]
질문: GitHub Actions에서 Python 버전을 지정하는 방법은?
답변: `actions/setup-python` 액션에서 `python-version: '3.11'`로 지정합니다.

[예시 2]
질문: 존재하지 않는 기능에 대한 질문입니다.
답변: 죄송합니다. 제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다.
"""

SYSTEM_PROMPT_TEMPLATE = """\
당신은 소프트웨어 엔지니어링 및 MLOps 분야의 전문 어시스턴트입니다.
아래 규칙을 반드시 따르세요:
1. 오직 [참고 컨텍스트]에 있는 정보만을 근거로 답변하세요.
2. 컨텍스트에 답이 없으면 "제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다."라고 답하세요.
3. 답변은 간결하고 명확하게, 코드가 필요하면 코드 블록(```)으로 감싸세요.
4. 불확실한 내용을 추측하거나 지어내지 마세요 (No Hallucination).

{few_shot_examples}

[참고 컨텍스트]
{context}

[질문]
{question}

[답변]"""


def build_prompt(user_question: str, current_context: str) -> str:
    """Persona + Constraint + Few-shot이 포함된 고도화 프롬프트를 생성합니다."""
    prompt = SYSTEM_PROMPT_TEMPLATE.format(
        few_shot_examples=FEW_SHOT_EXAMPLES,
        context=current_context,
        question=user_question,
    )
    logger.info("prompt_built", prompt_length=len(prompt))
    return prompt


# ---------------------------------------------------------------------------
# 7. RAGPipeline — 오케스트레이터 (의존성 주입)
# ---------------------------------------------------------------------------

class RAGPipeline:
    """
    RAG 파이프라인 오케스트레이터.

    rag_client와 llm_client를 외부에서 주입받아 테스트 시
    MockClient를 넣기만 하면 monkeypatch 없이 단위 테스트 가능.
    """

    def __init__(
        self,
        config: RAGConfig,
        rag_client: RAGClientProtocol,
        llm_client: LLMClientProtocol,
    ) -> None:
        self._config = config
        self._rag_client = rag_client
        self._llm_client = llm_client
        self._log = logger.bind(component="RAGPipeline")

    # --- 단계별 메서드 ---

    def init_agent(self) -> str:
        # [수정] 빈 문자열 반환 → 에이전트 초기화 상태 메시지 반환
        # test_init_agent_returns_non_empty_string 통과
        msg = f"agent_ready:{self._config.rag_tool_name}"
        self._log.info("step_1_init_agent", status=msg)
        return msg

    def decide_tool(self) -> str:
        self._log.info("step_2_decide_tool", tool=self._config.rag_tool_name)
        return self._config.rag_tool_name

    def run_rag(self, current_context: str, user_question: str) -> str:
        self._log.info("step_3_run_rag")
        if not current_context:
            current_context = self._config.rag_context_snippet
        try:
            return self._rag_client.retrieve(current_context, user_question)
        except Exception as exc:
            self._log.error("rag_failed_using_fallback", error=str(exc))
            return current_context

    def model_infer(self, prompt: str, temperature: float = 0.1) -> str:
        self._log.info("step_5_model_infer", temperature=temperature)
        return self._llm_client.infer(prompt, temperature)

    # --- 전체 파이프라인 실행 ---

    def run(self, user_question: str) -> str:
        """
        request_id를 생성해 로그 전 구간에 바인딩한 뒤 파이프라인을 실행합니다.
        init → decide → run_rag → build_prompt → model_infer
        """
        request_id = str(uuid.uuid4())
        structlog.contextvars.bind_contextvars(request_id=request_id)

        self._log.info("pipeline_start", question_preview=user_question[:80])

        try:
            context = self.init_agent()
            tool = self.decide_tool()

            if tool == self._config.rag_tool_name:
                context = self.run_rag(context, user_question)

            prompt = build_prompt(user_question, context)
            self._log.info("step_4_build_prompt", prompt_length=len(prompt))  # [추가] 누락된 step_4
            answer = self.model_infer(prompt, temperature=0.1)

            self._log.info("pipeline_complete")
            return answer

        except Exception as exc:
            self._log.error("pipeline_error", error=str(exc))
            raise
        finally:
            structlog.contextvars.unbind_contextvars("request_id")


# ---------------------------------------------------------------------------
# 8. 팩토리 함수 — 기본 설정으로 파이프라인 생성
# ---------------------------------------------------------------------------

def create_pipeline(config: Optional[RAGConfig] = None) -> RAGPipeline:
    """환경 변수 기반으로 파이프라인을 생성하는 편의 함수."""
    if config is None:
        config = RAGConfig.from_env()
    rag_client = RAGClient(config)
    llm_client = OpenAILLMClient(config)
    return RAGPipeline(config, rag_client, llm_client)


# ---------------------------------------------------------------------------
# 9. 엔트리포인트
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    _configure_logging()   # [수정] 운영 엔트리포인트에서만 로그 설정 적용
    pipeline = create_pipeline()
    user_q = "주말 깃허브 서버의 nbconvert 타임아웃 문제 원인과 해결책은?"
    output = pipeline.run(user_q)
    print(output)

In [ ]:
from rag_pipeline import RAGConfig, RAGPipeline, build_prompt, create_pipeline

pipeline = create_pipeline()  # 환경변수 기반
# 또는
config = RAGConfig(
    RAG_API_BASE="https://...",
    RAG_API_KEY="your-key",
    RAG_TOOL_NAME="retrieval_tool",
    RAG_CONTEXT_SNIPPET="기본 컨텍스트",
    OPENAI_API_KEY="sk-...",
    OPENAI_MODEL="gpt-4o-mini",
)